In [1]:
# Import Required Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import librosa
import soundfile as sf
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"Current working directory: {os.getcwd()}")

Libraries imported successfully!
Current working directory: e:\Research Hub\ASVDeepFake


In [2]:
# Define Dataset Paths
base_path = Path(r"e:/Research Hub/ASVDeepFake")
eval_data_path = base_path / "ASVspoof2021_DF_eval"
keys_path = base_path / "DF-keys-full" / "keys" / "DF"
audio_path = eval_data_path / "flac"

# Check if paths exist
print("Dataset Path Structure:")
print(f"Base path exists: {base_path.exists()}")
print(f"Evaluation data path exists: {eval_data_path.exists()}")
print(f"Keys path exists: {keys_path.exists()}")
print(f"Audio files path exists: {audio_path.exists()}")

# Count audio files
if audio_path.exists():
    audio_files = list(audio_path.glob("*.flac"))
    print(f"\nTotal audio files found: {len(audio_files)}")
    if audio_files:
        print(f"Sample filenames: {[f.name for f in audio_files[:5]]}")
else:
    print("Audio path not found!")

Dataset Path Structure:
Base path exists: True
Evaluation data path exists: True
Keys path exists: True
Audio files path exists: True

Total audio files found: 611829
Sample filenames: ['DF_E_2000011.flac', 'DF_E_2000013.flac', 'DF_E_2000024.flac', 'DF_E_2000026.flac', 'DF_E_2000027.flac']


In [4]:
# Load Trial List
trial_file = eval_data_path / "ASVspoof2021.DF.cm.eval.trl.txt"

if trial_file.exists():
    print(f"Loading trial list from: {trial_file}")
    
    # Read the trial list
    with open(trial_file, 'r') as f:
        trial_ids = [line.strip() for line in f.readlines()]
    
    print(f"Total trials: {len(trial_ids)}")
    print(f"Sample trial IDs: {trial_ids[:10]}")
    
    # Analyze trial ID patterns
    trial_df = pd.DataFrame({'trial_id': trial_ids})
    trial_df['prefix'] = trial_df['trial_id'].str[:4]  # Extract "DF_E"
    trial_df['number'] = trial_df['trial_id'].str[5:].astype(int)  # Extract number part
    
    print(f"\nTrial ID Analysis:")
    print(f"Prefix patterns: {trial_df['prefix'].value_counts()}")
    print(f"Number range: {trial_df['number'].min()} to {trial_df['number'].max()}")
    
else:
    print(f"Trial file not found: {trial_file}")
    trial_ids = []

Loading trial list from: e:\Research Hub\ASVDeepFake\ASVspoof2021_DF_eval_part00\ASVspoof2021_DF_eval\ASVspoof2021.DF.cm.eval.trl.txt
Total trials: 611829
Sample trial IDs: ['DF_E_2000011', 'DF_E_2000013', 'DF_E_2000024', 'DF_E_2000026', 'DF_E_2000027', 'DF_E_2000028', 'DF_E_2000031', 'DF_E_2000032', 'DF_E_2000040', 'DF_E_2000042']

Trial ID Analysis:
Prefix patterns: prefix
DF_E    611829
Name: count, dtype: int64
Number range: 2000011 to 4999993


In [5]:
# Analyze Audio Files
def analyze_audio_sample(file_path, max_samples=10):
    """Analyze a sample of audio files to understand their properties"""
    audio_info = []
    
    # Get a sample of audio files
    if audio_path.exists():
        sample_files = list(audio_path.glob("*.flac"))[:max_samples]
        
        print(f"Analyzing {len(sample_files)} sample audio files...")
        
        for i, audio_file in enumerate(sample_files):
            try:
                # Load audio info without loading the entire file
                info = sf.info(audio_file)
                
                audio_info.append({
                    'filename': audio_file.name,
                    'duration': info.duration,
                    'sample_rate': info.samplerate,
                    'channels': info.channels,
                    'frames': info.frames
                })
                
                print(f"File {i+1}/{len(sample_files)}: {audio_file.name} - "
                      f"{info.duration:.2f}s, {info.samplerate}Hz")
                
            except Exception as e:
                print(f"Error analyzing {audio_file.name}: {e}")
        
        return pd.DataFrame(audio_info)
    else:
        print("Audio path not found!")
        return pd.DataFrame()

# Run audio analysis
audio_df = analyze_audio_sample(audio_path, max_samples=20)

if not audio_df.empty:
    print(f"\nAudio File Statistics:")
    print(f"Duration range: {audio_df['duration'].min():.2f}s to {audio_df['duration'].max():.2f}s")
    print(f"Average duration: {audio_df['duration'].mean():.2f}s")
    print(f"Sample rates: {audio_df['sample_rate'].unique()}")
    print(f"Channels: {audio_df['channels'].unique()}")
else:
    print("No audio files analyzed.")

Analyzing 20 sample audio files...
File 1/20: DF_E_2000011.flac - 1.43s, 16000Hz
File 2/20: DF_E_2000013.flac - 3.20s, 16000Hz
File 3/20: DF_E_2000024.flac - 1.86s, 16000Hz
File 4/20: DF_E_2000026.flac - 2.94s, 16000Hz
File 5/20: DF_E_2000027.flac - 1.86s, 16000Hz
File 6/20: DF_E_2000028.flac - 1.51s, 16000Hz
File 7/20: DF_E_2000031.flac - 3.01s, 16000Hz
File 8/20: DF_E_2000032.flac - 3.90s, 16000Hz
File 9/20: DF_E_2000040.flac - 3.14s, 16000Hz
File 10/20: DF_E_2000042.flac - 4.35s, 16000Hz
File 11/20: DF_E_2000044.flac - 2.74s, 16000Hz
File 12/20: DF_E_2000048.flac - 2.94s, 16000Hz
File 13/20: DF_E_2000049.flac - 1.41s, 16000Hz
File 14/20: DF_E_2000053.flac - 3.97s, 16000Hz
File 15/20: DF_E_2000055.flac - 4.58s, 16000Hz
File 16/20: DF_E_2000058.flac - 1.73s, 16000Hz
File 17/20: DF_E_2000072.flac - 4.71s, 16000Hz
File 18/20: DF_E_2000075.flac - 3.14s, 16000Hz
File 19/20: DF_E_2000079.flac - 2.69s, 16000Hz
File 20/20: DF_E_2000080.flac - 2.25s, 16000Hz

Audio File Statistics:
Duration r

In [6]:
# Explore Baseline System Results
baseline_systems = ['CQCC-GMM', 'LFCC-GMM', 'LFCC-LCNN', 'RawNet2']

print("Available Baseline Systems:")
for system in baseline_systems:
    system_path = keys_path / "CM" / system
    if system_path.exists():
        print(f"✓ {system}: {system_path}")
        
        # Check for score files
        score_file = system_path / "score.txt"
        if score_file.exists():
            print(f"  - Score file found: {score_file}")
            try:
                # Read first few lines to understand format
                with open(score_file, 'r') as f:
                    sample_lines = [f.readline().strip() for _ in range(5)]
                print(f"  - Sample score format: {sample_lines[:3]}")
            except Exception as e:
                print(f"  - Error reading score file: {e}")
        else:
            print(f"  - No score file found")
    else:
        print(f"✗ {system}: Not found")

# Check trial metadata
metadata_file = keys_path / "CM" / "trial_metadata.txt"
if metadata_file.exists():
    print(f"\nTrial metadata file found: {metadata_file}")
    print(f"File size: {metadata_file.stat().st_size / (1024*1024):.1f} MB")
    
    # Try to read a few lines to understand the format
    try:
        with open(metadata_file, 'r') as f:
            header = f.readline().strip()
            sample_lines = [f.readline().strip() for _ in range(5)]
        
        print(f"Header: {header}")
        print(f"Sample entries: {sample_lines[:3]}")
    except Exception as e:
        print(f"Error reading metadata file: {e}")
else:
    print(f"\nTrial metadata file not found: {metadata_file}")

Available Baseline Systems:
✓ CQCC-GMM: e:\Research Hub\ASVDeepFake\DF-keys-full\keys\DF\CM\CQCC-GMM
  - Score file found: e:\Research Hub\ASVDeepFake\DF-keys-full\keys\DF\CM\CQCC-GMM\score.txt
  - Sample score format: ['DF_E_2000011 0.419864', 'DF_E_2000013 0.524628', 'DF_E_2000024 0.193270']
✓ LFCC-GMM: e:\Research Hub\ASVDeepFake\DF-keys-full\keys\DF\CM\LFCC-GMM
  - Score file found: e:\Research Hub\ASVDeepFake\DF-keys-full\keys\DF\CM\LFCC-GMM\score.txt
  - Sample score format: ['DF_E_2000011 1.476069', 'DF_E_2000013 1.374222', 'DF_E_2000024 -0.466819']
✓ LFCC-LCNN: e:\Research Hub\ASVDeepFake\DF-keys-full\keys\DF\CM\LFCC-LCNN
  - Score file found: e:\Research Hub\ASVDeepFake\DF-keys-full\keys\DF\CM\LFCC-LCNN\score.txt
  - Sample score format: ['DF_E_2000011 -11.968538', 'DF_E_2000013 -3.396158', 'DF_E_2000024 -15.476466']
✓ RawNet2: e:\Research Hub\ASVDeepFake\DF-keys-full\keys\DF\CM\RawNet2
  - Score file found: e:\Research Hub\ASVDeepFake\DF-keys-full\keys\DF\CM\RawNet2\score.txt

In [7]:
# Audio Visualization Functions
def plot_audio_analysis(audio_file_path, figsize=(15, 10)):
    """Plot waveform and spectrogram for an audio file"""
    try:
        # Load audio
        y, sr = librosa.load(audio_file_path, sr=None)
        
        # Create subplots
        fig, axes = plt.subplots(3, 1, figsize=figsize)
        
        # Plot waveform
        axes[0].plot(np.linspace(0, len(y)/sr, len(y)), y)
        axes[0].set_title(f'Waveform - {audio_file_path.name}')
        axes[0].set_xlabel('Time (s)')
        axes[0].set_ylabel('Amplitude')
        axes[0].grid(True, alpha=0.3)
        
        # Plot spectrogram
        D = librosa.stft(y)
        S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
        img = librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz', ax=axes[1])
        axes[1].set_title('Spectrogram')
        plt.colorbar(img, ax=axes[1], format='%+2.0f dB')
        
        # Plot mel spectrogram
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        S_db_mel = librosa.amplitude_to_db(S, ref=np.max)
        img2 = librosa.display.specshow(S_db_mel, sr=sr, x_axis='time', y_axis='mel', ax=axes[2])
        axes[2].set_title('Mel Spectrogram')
        plt.colorbar(img2, ax=axes[2], format='%+2.0f dB')
        
        plt.tight_layout()
        plt.show()
        
        # Print audio statistics
        print(f"Audio Statistics for {audio_file_path.name}:")
        print(f"Duration: {len(y)/sr:.2f} seconds")
        print(f"Sample Rate: {sr} Hz")
        print(f"Total Samples: {len(y)}")
        print(f"RMS Energy: {np.sqrt(np.mean(y**2)):.4f}")
        print(f"Max Amplitude: {np.max(np.abs(y)):.4f}")
        
    except Exception as e:
        print(f"Error analyzing {audio_file_path}: {e}")

# Select a sample audio file for visualization
if audio_path.exists():
    sample_files = list(audio_path.glob("*.flac"))
    if sample_files:
        sample_file = sample_files[0]  # Take the first file
        print(f"Analyzing sample file: {sample_file.name}")
        print("Note: This will load and process the audio file, which may take a moment...")
        
        # Uncomment the line below to run the visualization
        # plot_audio_analysis(sample_file)
        print("To visualize the audio, uncomment the plot_audio_analysis() call above")
    else:
        print("No audio files found for visualization")
else:
    print("Audio path not found")

Analyzing sample file: DF_E_2000011.flac
Note: This will load and process the audio file, which may take a moment...
To visualize the audio, uncomment the plot_audio_analysis() call above
